# Machine Learning Classification Tool

An interactive classification notebook that implements three ML algorithms from scratch using only Python built-ins and NumPy — no scikit-learn for the core models.

**Concepts covered:** Functions · Iteration · Strings · File I/O · Lists · Dictionaries · Tuples


In [ ]:
import csv
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

try:
    from google.colab import data_table
    from google.colab import drive
    IN_COLAB = True
except Exception:
    data_table = None
    drive = None
    IN_COLAB = False

try:
    from IPython.display import display
except Exception:
    display = print

# Keep the same libraries, and only prepare the save folder.
if IN_COLAB:
    drive.mount('/content/drive')
    SAVE_FOLDER = '/content/drive/MyDrive/ColabData'
else:
    SAVE_FOLDER = os.path.join(os.getcwd(), 'ColabData')

os.makedirs(SAVE_FOLDER, exist_ok=True)
print('Setup is ready.')
print('Save folder:', SAVE_FOLDER)


## Step 1 — Load Data

Reads a CSV file from a local path or a public Google Drive link. The first row is treated as column headers; remaining rows are loaded as a NumPy array.

In [ ]:
# STEP 1 - Load data from a CSV file

def extract_drive_id(link):
    # Get the file ID from a common Google Drive link.
    if 'id=' in link:
        return link.split('id=')[1].split('&')[0]

    parts = link.split('/')
    for i in range(len(parts)):
        if parts[i] == 'd' and i + 1 < len(parts):
            return parts[i + 1]
    return None


def download_drive_file(link, save_path):
    # Download a public CSV file from Google Drive.
    file_id = extract_drive_id(link)
    if file_id is None:
        print('Invalid Google Drive link.')
        return None

    url = 'https://drive.google.com/uc?export=download&id=' + file_id
    print('Downloading file...')

    try:
        urllib.request.urlretrieve(url, save_path)
        print('File downloaded.')
        return save_path
    except Exception as error:
        print('Download failed:', error)
        return None


def read_csv_file(file_path):
    # Read a CSV file using the csv module.
    rows = None

    for file_encoding in ['utf-8-sig', 'latin-1']:
        try:
            file_handle = open(file_path, 'r', encoding=file_encoding, newline='')
            rows = []
            reader = csv.reader(file_handle)

            for row in reader:
                clean_row = []
                for value in row:
                    clean_row.append(value.strip())
                if len(clean_row) > 0 and any(value != '' for value in clean_row):
                    rows.append(clean_row)

            file_handle.close()
            break
        except UnicodeDecodeError:
            file_handle.close()
            rows = None
        except Exception as error:
            print('File error:', error)
            return None, None

    if rows is None:
        print('File encoding is not supported.')
        return None, None

    if len(rows) < 2:
        print('The CSV file must have headers and data rows.')
        return None, None

    headers = rows[0]
    width = len(headers)
    data_rows = []

    for row in rows[1:]:
        while len(row) < width:
            row.append('')
        data_rows.append(row[:width])

    data = np.array(data_rows, dtype=object)
    print('File loaded successfully.')
    print('Rows:', data.shape[0], 'Columns:', data.shape[1])
    return headers, data


# Sample dataset (copy and paste the link below):
# https://drive.google.com/file/d/16AMnMYPwkpWCCO6cZEWzXfUC_fyvnonO/view?usp=sharing
user_input = input('Enter CSV path or Google Drive link: ').strip()

if 'drive.google.com' in user_input:
    local_file = os.path.join(SAVE_FOLDER, 'downloaded_data.csv')
    file_path = download_drive_file(user_input, local_file)
else:
    file_path = user_input

headers, raw_data = read_csv_file(file_path)


## Step 2 — Clean Data

Removes duplicate rows, then fills missing values column by column:
- **Numeric columns** → filled with the column mean
- **Text columns** → filled with the most frequent value

The cleaned dataset is saved as `cleaned_data.csv`.

In [ ]:
# STEP 2 - Clean data and show simple statistics

MISSING_VALUES = ['', 'nan', 'null', 'none', '?']


def is_missing(value):
    # Check if a value is empty or missing.
    text = str(value).strip().lower()
    return text in MISSING_VALUES


def is_number(value):
    # Check if a value can be converted to a number.
    if is_missing(value):
        return False
    try:
        float(value)
        return True
    except Exception:
        return False


def remove_duplicate_rows(data):
    # Use a list and a set to keep only unique rows.
    unique_rows = []
    seen_rows = set()

    for row in data:
        key = tuple(row)
        if key not in seen_rows:
            seen_rows.add(key)
            unique_rows.append(row)

    return np.array(unique_rows, dtype=object)


def most_common_value(values):
    # Count text values using a dictionary.
    counts = {}
    for value in values:
        if not is_missing(value):
            counts[value] = counts.get(value, 0) + 1

    if len(counts) == 0:
        return 'unknown'

    best_value = None
    best_count = -1
    for value in counts:
        if counts[value] > best_count:
            best_value = value
            best_count = counts[value]
    return best_value


def fill_missing_values(data):
    # Fill empty cells column by column.
    filled_count = 0

    for col in range(data.shape[1]):
        column = data[:, col]
        available = []

        for value in column:
            if not is_missing(value):
                available.append(value)

        numeric_column = len(available) > 0
        for value in available:
            if not is_number(value):
                numeric_column = False

        if numeric_column:
            numbers = []
            for value in available:
                numbers.append(float(value))
            fill_value = str(round(sum(numbers) / len(numbers), 4))
        else:
            fill_value = most_common_value(available)

        for row in range(data.shape[0]):
            if is_missing(data[row, col]):
                data[row, col] = fill_value
                filled_count = filled_count + 1

    return data, filled_count


def save_clean_file(headers, data):
    # Save the cleaned dataset as a new CSV file.
    clean_path = os.path.join(SAVE_FOLDER, 'cleaned_data.csv')
    file_handle = open(clean_path, 'w', encoding='utf-8', newline='')
    writer = csv.writer(file_handle)
    writer.writerow(headers)
    for row in data:
        writer.writerow(list(row))
    file_handle.close()
    print('Clean file saved:', clean_path)


def clean_data(headers, data):
    if data is None:
        raise ValueError('No data was loaded.')

    old_rows = data.shape[0]
    data = remove_duplicate_rows(data)
    new_rows = data.shape[0]
    data, filled_count = fill_missing_values(data)

    print('Data cleaning is done.')
    print('Duplicate rows removed:', old_rows - new_rows)
    print('Missing values filled:', filled_count)
    print('Rows after cleaning:', data.shape[0])
    save_clean_file(headers, data)
    return data


def show_statistics(headers, data):
    # Show basic statistics for numeric columns.
    rows = []

    for col in range(len(headers)):
        numbers = []
        for value in data[:, col]:
            if is_number(value):
                numbers.append(float(value))

        if len(numbers) > 0:
            arr = np.array(numbers)
            rows.append({
                'Column': headers[col],
                'Min': round(float(np.min(arr)), 4),
                'Mean': round(float(np.mean(arr)), 4),
                'Std': round(float(np.std(arr)), 4),
                'Max': round(float(np.max(arr)), 4)
            })

    print('\nNumeric column statistics:')
    if len(rows) == 0:
        print('No numeric columns found.')
    elif data_table is not None:
        table_data = {}
        for key in ['Column', 'Min', 'Mean', 'Std', 'Max']:
            table_data[key] = []
            for row in rows:
                table_data[key].append(row[key])
        display(data_table.DataTable(table_data, num_rows_per_page=15))
    else:
        for row in rows:
            print(row)


clean = clean_data(headers, raw_data)
show_statistics(headers, clean)


## Step 3 — Select Features & Encode

The user selects the target column (class label) and feature columns interactively.
Text values are encoded to integers using a dictionary map, e.g. `{'Male': 0, 'Female': 1}`.

In [ ]:
# STEP 3 - Select features and target

def show_columns(headers):
    print('\nColumns:')
    for i in range(len(headers)):
        print(str(i) + ' - ' + headers[i])


def ask_column_number(message, headers):
    while True:
        user_value = input(message).strip()
        try:
            index = int(user_value)
            if index >= 0 and index < len(headers):
                return index
            print('Please enter a number from 0 to', len(headers) - 1)
        except Exception:
            print('Please enter a valid number.')


def parse_feature_columns(text, target_index, headers):
    # Empty input means all columns except the target column.
    if text.strip() == '':
        features = []
        for i in range(len(headers)):
            if i != target_index:
                features.append(i)
        return features

    features = []
    parts = text.split(',')

    for part in parts:
        part = part.strip()
        try:
            index = int(part)
            if index == target_index:
                print('Target column skipped:', index)
            elif index >= 0 and index < len(headers):
                if index not in features:
                    features.append(index)
            else:
                print('Column number skipped:', index)
        except Exception:
            print('Invalid column skipped:', part)

    if len(features) == 0:
        raise ValueError('No feature columns selected.')
    return features


def encode_text_values(values):
    # Convert text values to numbers using a dictionary.
    mapping = {}
    encoded = []
    next_number = 0

    for value in values:
        text = str(value)
        if text not in mapping:
            mapping[text] = next_number
            next_number = next_number + 1
        encoded.append(mapping[text])

    return np.array(encoded), mapping


def convert_feature_column(values):
    # Numeric columns stay numeric. Text columns are encoded.
    all_numbers = True
    for value in values:
        if not is_number(value):
            all_numbers = False

    if all_numbers:
        numbers = []
        for value in values:
            numbers.append(float(value))
        return np.array(numbers), None

    encoded, mapping = encode_text_values(values)
    return encoded.astype(float), mapping


def prepare_data(headers, data):
    show_columns(headers)
    target_index = ask_column_number('\nEnter target column number: ', headers)

    y, target_mapping = encode_text_values(data[:, target_index])
    print('Target mapping:', target_mapping)

    print('\nEnter feature columns separated by commas.')
    print('Press Enter to use all columns except target.')
    feature_text = input('Feature columns: ').strip()
    feature_indices = parse_feature_columns(feature_text, target_index, headers)

    feature_arrays = []
    feature_maps = {}
    selected_names = []

    for index in feature_indices:
        column_values, mapping = convert_feature_column(data[:, index])
        feature_arrays.append(column_values)
        selected_names.append(headers[index])
        if mapping is not None:
            feature_maps[headers[index]] = mapping

    X = np.column_stack(feature_arrays).astype(float)
    y = y.astype(int)

    print('Data is ready for training.')
    print('Feature shape:', X.shape)
    print('Target shape:', y.shape)
    print('Selected features:', selected_names)

    return X, y, selected_names, headers[target_index], target_mapping, feature_maps


X, y, selected_features, target_name, target_mapping, feature_maps = prepare_data(headers, clean)


## Step 4 — Split & Scale

Data is split into training and test sets (default 80/20). Features are standardized using
train-set statistics only (zero-mean, unit-variance) to prevent data leakage.

In [ ]:
# STEP 4 - Train/test split and simple scaling

def train_test_split_simple(X, y, test_ratio=0.2):
    # Shuffle indices, then split them into train and test.
    test_ratio = float(test_ratio)
    if test_ratio < 0.1:
        test_ratio = 0.1
    if test_ratio > 0.5:
        test_ratio = 0.5

    np.random.seed(42)
    indices = np.arange(len(y))
    np.random.shuffle(indices)

    test_size = int(round(len(y) * test_ratio))
    if test_size < 1:
        test_size = 1
    if test_size >= len(y):
        test_size = len(y) - 1

    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]


def standardize_data(X_train, X_test):
    # Use train mean and std only.
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)

    for i in range(len(std)):
        if std[i] == 0:
            std[i] = 1

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    return X_train_scaled, X_test_scaled


ratio = input('Enter test ratio, default is 0.2: ').strip()
if ratio == '':
    ratio = '0.2'

X_train, X_test, y_train, y_test = train_test_split_simple(X, y, ratio)
X_train_scaled, X_test_scaled = standardize_data(X_train, X_test)

print('Split is done.')
print('Train rows:', len(y_train))
print('Test rows:', len(y_test))


## Step 5 — Algorithms

Three classifiers implemented from scratch:

1. **K-Nearest Neighbors** — classifies each test row by majority vote among its *k* nearest training neighbors (Euclidean distance).
2. **Gaussian Naive Bayes** — estimates per-class mean and std for each feature, then picks the highest log-posterior class.
3. **Decision Tree** — builds a small recursive tree (max depth 3) using Gini impurity as the split criterion.

In [ ]:
# STEP 5 - Simple algorithms using functions, lists, dictionaries, and tuples

def majority_vote(labels):
    # Return the most repeated label using a dictionary.
    counts = {}
    for label in labels:
        label = int(label)
        counts[label] = counts.get(label, 0) + 1

    best_label = None
    best_count = -1
    for label in counts:
        if counts[label] > best_count:
            best_label = label
            best_count = counts[label]
    return best_label


# Algorithm 1: K-Nearest Neighbors
def predict_knn_one(X_train, y_train, test_row, k=5):
    distances = []

    for i in range(len(y_train)):
        distance = np.sqrt(np.sum((X_train[i] - test_row) ** 2))
        distances.append((distance, int(y_train[i])))  # tuple: (distance, label)

    distances.sort(key=lambda item: item[0])

    if k > len(distances):
        k = len(distances)
    if k % 2 == 0 and k > 1:
        k = k - 1

    nearest_labels = []
    for i in range(k):
        nearest_labels.append(distances[i][1])

    return majority_vote(nearest_labels)


def predict_knn(X_train, y_train, X_test, k=5):
    predictions = []
    for row in X_test:
        predictions.append(predict_knn_one(X_train, y_train, row, k))
    return np.array(predictions)


# Algorithm 2: Gaussian Naive Bayes
def train_naive_bayes(X_train, y_train):
    model = {}
    label_values = np.unique(y_train)

    for label_value in label_values:
        rows = X_train[y_train == label_value]
        model[int(label_value)] = {
            'prior': len(rows) / len(X_train),
            'mean': np.mean(rows, axis=0),
            'std': np.std(rows, axis=0) + 0.000001
        }

    return model


def predict_naive_bayes_one(model, row):
    best_label = None
    best_score = None

    for label_value in model:
        info = model[label_value]
        score = np.log(info['prior'])

        for i in range(len(row)):
            mean = info['mean'][i]
            std = info['std'][i]
            x = row[i]
            probability = (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-((x - mean) ** 2) / (2 * std ** 2))
            score = score + np.log(probability + 0.000001)

        if best_score is None or score > best_score:
            best_score = score
            best_label = label_value

    return best_label


def predict_naive_bayes(model, X_test):
    predictions = []
    for row in X_test:
        predictions.append(predict_naive_bayes_one(model, row))
    return np.array(predictions)


# Algorithm 3: Simple Decision Tree
def gini(labels):
    counts = {}
    for label in labels:
        label = int(label)
        counts[label] = counts.get(label, 0) + 1

    score = 1
    for label in counts:
        probability = counts[label] / len(labels)
        score = score - probability ** 2
    return score


def split_data(X, y, feature_index, threshold):
    left_X = []
    left_y = []
    right_X = []
    right_y = []

    for i in range(len(y)):
        if X[i][feature_index] <= threshold:
            left_X.append(X[i])
            left_y.append(y[i])
        else:
            right_X.append(X[i])
            right_y.append(y[i])

    return np.array(left_X), np.array(left_y), np.array(right_X), np.array(right_y)


def possible_thresholds(values):
    unique_values = sorted(list(set(values)))
    thresholds = []

    for i in range(len(unique_values) - 1):
        middle = (unique_values[i] + unique_values[i + 1]) / 2
        thresholds.append(middle)

    if len(thresholds) > 20:
        step = max(1, len(thresholds) // 20)
        thresholds = thresholds[::step]

    return thresholds


def best_split(X, y):
    best_feature = None
    best_threshold = None
    best_score = gini(y)

    for feature_index in range(X.shape[1]):
        thresholds = possible_thresholds(X[:, feature_index])

        for threshold in thresholds:
            left_X, left_y, right_X, right_y = split_data(X, y, feature_index, threshold)

            if len(left_y) == 0 or len(right_y) == 0:
                continue

            left_score = gini(left_y)
            right_score = gini(right_y)
            total_score = (len(left_y) / len(y)) * left_score + (len(right_y) / len(y)) * right_score

            if total_score < best_score:
                best_score = total_score
                best_feature = feature_index
                best_threshold = threshold

    return best_feature, best_threshold


def build_tree(X, y, depth=0, max_depth=3):
    # A small recursive tree that returns a dictionary.
    if depth >= max_depth or len(set(y)) == 1 or len(y) < 4:
        return {'type': 'leaf', 'label': majority_vote(y)}

    feature_index, threshold = best_split(X, y)
    if feature_index is None:
        return {'type': 'leaf', 'label': majority_vote(y)}

    left_X, left_y, right_X, right_y = split_data(X, y, feature_index, threshold)

    return {
        'type': 'node',
        'feature': feature_index,
        'threshold': threshold,
        'left': build_tree(left_X, left_y, depth + 1, max_depth),
        'right': build_tree(right_X, right_y, depth + 1, max_depth)
    }


def predict_tree_one(tree, row):
    if tree['type'] == 'leaf':
        return tree['label']

    feature_index = tree['feature']
    threshold = tree['threshold']

    if row[feature_index] <= threshold:
        return predict_tree_one(tree['left'], row)
    return predict_tree_one(tree['right'], row)


def predict_tree(tree, X_test):
    predictions = []
    for row in X_test:
        predictions.append(predict_tree_one(tree, row))
    return np.array(predictions)


print('Algorithms are ready.')
print('Models: KNN, Gaussian Naive Bayes, Simple Decision Tree')


## Step 6 — Evaluation

Each model is evaluated with accuracy and a seaborn confusion matrix.
A horizontal bar chart compares all three models side by side.

In [ ]:
# STEP 6 - Evaluation and charts

def accuracy(y_true, y_pred):
    correct = 0
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct = correct + 1
    return correct / len(y_true) * 100


def get_target_names(target_mapping):
    # Convert {'name': number} into a list ordered by number.
    names = [''] * len(target_mapping)
    for name in target_mapping:
        index = target_mapping[name]
        names[index] = name
    return names


def show_confusion_matrix(y_true, y_pred, model_name):
    labels = list(range(len(target_mapping)))
    label_names = get_target_names(target_mapping)
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    model_accuracy = accuracy(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(max(5, len(labels) * 1.4), max(4, len(labels) * 1.2)))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        linewidths=0.5,
        linecolor='#1a2634',
        xticklabels=label_names,
        yticklabels=label_names,
        ax=ax
    )

    ax.set_xlabel('Predicted', color='white')
    ax.set_ylabel('Actual', color='white')
    ax.set_title(model_name + ' - Accuracy: ' + str(round(model_accuracy, 2)) + '%', color='white')
    ax.tick_params(colors='white')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', color='white')
    plt.setp(ax.get_yticklabels(), rotation=0, color='white')
    plt.tight_layout()
    plt.show()

    print('Correct predictions:', int(np.sum(y_true == y_pred)), '/', len(y_true))
    print('Accuracy:', round(model_accuracy, 2), '%')


def show_comparison_chart(results):
    names = list(results.keys())
    values = list(results.values())

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    bars = ax.barh(names, values, color=['#2d9cdb', '#27ae60', '#f2c94c'])

    for bar, value in zip(bars, values):
        ax.text(value + 0.7, bar.get_y() + bar.get_height() / 2, str(round(value, 1)) + '%',
                va='center', color='white')

    ax.set_xlim(0, 105)
    ax.set_xlabel('Accuracy %', color='white')
    ax.set_title('Algorithm Comparison', color='white')
    ax.tick_params(colors='white')
    plt.tight_layout()
    plt.show()


print('Evaluation tools are ready.')


## Step 7 — Main Menu

An interactive `while` loop lets the user run any single model or compare all three at once.

In [ ]:
# STEP 7 - Main menu

def run_model(choice):
    if choice == '1':
        name = 'K-Nearest Neighbors'
        predictions = predict_knn(X_train_scaled, y_train, X_test_scaled, k=5)

    elif choice == '2':
        name = 'Gaussian Naive Bayes'
        model = train_naive_bayes(X_train_scaled, y_train)
        predictions = predict_naive_bayes(model, X_test_scaled)

    elif choice == '3':
        name = 'Simple Decision Tree'
        tree = build_tree(X_train, y_train, max_depth=3)
        predictions = predict_tree(tree, X_test)

    show_confusion_matrix(y_test, predictions, name)
    return name, accuracy(y_test, predictions)


def compare_all_models():
    print('Running all models...')
    results = {}

    name, score = run_model('1')
    results[name] = score

    name, score = run_model('2')
    results[name] = score

    name, score = run_model('3')
    results[name] = score

    show_comparison_chart(results)

    best_name = None
    best_score = -1
    for name in results:
        if results[name] > best_score:
            best_name = name
            best_score = results[name]

    print('\nFinal results:')
    for name in results:
        print(name + ':', round(results[name], 2), '%')
    print('Best model:', best_name, '-', round(best_score, 2), '%')


def main_menu():
    while True:
        print('\n' + '=' * 42)
        print('        ML Classification Tool')
        print('=' * 42)
        print('1. K-Nearest Neighbors')
        print('2. Gaussian Naive Bayes')
        print('3. Simple Decision Tree')
        print('4. Compare All Three')
        print('0. Exit')
        print('=' * 42)

        choice = input('Choose an option: ').strip()

        if choice == '1' or choice == '2' or choice == '3':
            run_model(choice)
        elif choice == '4':
            compare_all_models()
        elif choice == '0':
            print('Goodbye.')
            break
        else:
            print('Invalid choice. Try again.')


main_menu()
